In [1]:
# ==============================================================================
# 1. INSTALLS & IMPORTS
# ==============================================================================
!pip install trafilatura requests ddgs

import os, re, shutil, time, requests, urllib.parse, random, datetime
from pathlib import Path
import xml.etree.ElementTree as ET
import trafilatura
from ddgs import DDGS

# ==============================================================================
# 2. PATH MAPPING & CONFIGURATION
# ==============================================================================
# IMPORTANT: Point this to your local Google Drive sync folder (e.g., G: drive)
DRIVE_DIR = Path("G:/My Drive/research-assistant")

DRIVE_WORKSPACE = DRIVE_DIR / "workspace"
DRIVE_PERSONALITY = DRIVE_DIR / "personality"
DRIVE_UNPROCESSED = DRIVE_WORKSPACE / "unprocessed"
DRIVE_QUERIES = DRIVE_PERSONALITY / "queries"
DRIVE_PENDING = DRIVE_QUERIES / "pending"    # cloud writes, forager consumes
DRIVE_CLAIMED = DRIVE_QUERIES / "claimed"    # forager: in progress
DRIVE_DONE = DRIVE_QUERIES / "done"          # forager: finished (this is the ledger)
DRIVE_PAPERS_LEDGER = DRIVE_QUERIES / "foraged_papers_ledger.md"   # forager-only, append-only
DRIVE_HARVESTED = DRIVE_QUERIES / "harvested_keywords.md"           # forager-only, append-only
DRIVE_NUDGE = DRIVE_PERSONALITY / "NUDGE.md"                         # optional; its words are the anchor candidates

LOCAL_WORKSPACE = Path("./research-workspace").resolve()
LOCAL_PERSONALITY = Path("./research-personality").resolve()
LOCAL_UNPROCESSED = LOCAL_WORKSPACE / "unprocessed"
LOCAL_QUERIES = LOCAL_PERSONALITY / "queries"
LOCAL_PAPERS_LEDGER = LOCAL_QUERIES / "foraged_papers_ledger.md"    # this run's additions; appended to DRIVE_PAPERS_LEDGER
LOCAL_HARVESTED = LOCAL_QUERIES / "harvested_keywords.md"           # this run's additions; appended to DRIVE_HARVESTED

directories = [LOCAL_WORKSPACE, LOCAL_PERSONALITY, LOCAL_UNPROCESSED, LOCAL_QUERIES, 
               DRIVE_WORKSPACE, DRIVE_PERSONALITY, DRIVE_UNPROCESSED, DRIVE_QUERIES,
               DRIVE_PENDING, DRIVE_CLAIMED, DRIVE_DONE]
for p in directories: p.mkdir(parents=True, exist_ok=True)

LOCAL_API_KEYS = Path("./API_KEYS.md").resolve()
def load_api_keys(path: Path) -> dict:
    """Reads NAME = value lines from a markdown file; every other line is ignored."""
    try:
        text = path.read_text(encoding="utf-8")
    except OSError:
        print(f"[!] Could not read API keys from {path}. Keyed sources will run without keys.")
        return {}
    keys = {}
    for line in text.splitlines():
        m = re.match(r"^\s*([A-Z][A-Z0-9_]*)\s*=\s*(.*?)\s*$", line)
        if m: keys[m.group(1)] = m.group(2).strip("\"'` ")
    return keys

API_KEYS = load_api_keys(LOCAL_API_KEYS)

OPENALEX_API_KEY = API_KEYS.get("OPENALEX_API_KEY", "")
S2_API_KEY = API_KEYS.get("S2_API_KEY", "")
CORE_API_KEY = API_KEYS.get("CORE_API_KEY", "")
ZENODO_API_KEY = API_KEYS.get("ZENODO_API_KEY", "")

CLAIM_TIMEOUT_HOURS = 2   # a claim older than this is assumed to belong to a crashed run
MAX_QUERIES_PER_RUN = None              # None = keep going until the queue is empty; a number stops sooner
LIMIT = 2 #results saved per source, per query (e.g., 2 from Zenodo, 2 from CORE, 2 from arXiv, etc)

# SEARCH WINDOW: publication years, applied to every source that can filter by year
YEAR_FROM = 2021   # first publication year to include
YEAR_TO = None     # last year to include; None = next calendar year (keeps "forthcoming" items)

# ==============================================================================
# 3. WORKER FUNCTIONS
# ==============================================================================
def year_bounds():
    """The (first, last) publication year to search, from YEAR_FROM / YEAR_TO."""
    start = int(YEAR_FROM) if YEAR_FROM else 1900
    end = int(YEAR_TO) if YEAR_TO else datetime.datetime.now().year + 1
    return (end, start) if start > end else (start, end)     # tolerate swapped values

def parse_year(value):
    """Best-effort year from 2024, '2024', '2024-05-01T00:00:00Z' ... -> int, or None if unknown."""
    if value is None: return None
    m = re.match(r"\s*(\d{4})", str(value))
    return int(m.group(1)) if m else None

def year_in_range(value) -> bool:
    """Client-side safety net. An unknown year is kept rather than discarded."""
    year = parse_year(value)
    if year is None: return True
    start, end = year_bounds()
    return start <= year <= end

def generate_safe_title(raw_title: str) -> str:
    safe_string = "".join([c for c in raw_title if c.isalnum() or c == ' '])
    return safe_string[:120].rstrip()

def reconstruct_abstract(inverted_index) -> str:
    """Rebuilds plain text from an OpenAlex `abstract_inverted_index` ({word: [positions]})."""
    if not isinstance(inverted_index, dict) or not inverted_index:
        return ""
    try:
        words = {}
        for word, positions in inverted_index.items():
            for pos in positions:
                words[pos] = word
        return " ".join(words[i] for i in sorted(words)).strip()
    except Exception:
        return ""

def load_nudge_words(path: Path) -> list:
    """Anchor candidates from NUDGE.md. A missing or empty file gives []. Lines starting with # are ignored."""
    try:
        text = path.read_text(encoding="utf-8")
    except OSError:
        return []
    lines = [l for l in text.splitlines() if not l.lstrip().startswith("#")]
    return list(dict.fromkeys(w.lower() for w in re.findall(r"\w[\w'-]*", "\n".join(lines))))
# ==============================================================================
# 4. METADATA HARVESTING
# ==============================================================================
def harvest_metadata_concepts(work_data: dict, source_name: str = "OpenAlex"):
    """Extracts raw JSON ontology data from APIs to feed back to the Colab Brain."""
    extracted_terms = []
    if source_name == "OpenAlex":
        for concept in work_data.get('concepts', []):
            if concept.get('score', 0) > 0.4: extracted_terms.append(concept.get('display_name'))
    elif source_name == "Semantic Scholar":
        for field in work_data.get('fieldsOfStudy', []) or []: extracted_terms.append(field)
    
    clean_terms = list(set([t.strip() for t in extracted_terms if t and len(t.strip()) > 2]))
    if clean_terms:
        with open(LOCAL_HARVESTED, "a", encoding="utf-8") as f:
            f.write(f"- **{work_data.get('title', 'Unknown')}** [{source_name}]: {', '.join(clean_terms)}\n")
    return clean_terms

# ==============================================================================
# 5. DEAD-DROP QUEUE  (one query per file; and a file's location indicates where the query it contains is, in the workflow)
#     pending/  -> finished creating query          claimed/ -> in process of searching
#     done/     -> finished searching
#     Because there is shared access to these files, they are never overwritten.
# ==============================================================================

def _retry(fn, attempts: int = 4, delay: int = 2):
    """Retries filesystem calls that fail because Google Drive Desktop briefly locks a file."""
    for i in range(attempts):
        try:
            return fn()
        except FileNotFoundError:
            raise
        except OSError:
            if i == attempts - 1: raise
            print("[*] File lock detected by Google Drive sync. Retrying in 2s...")
            time.sleep(delay)

def requeue_stale_claims(max_age_hours: float = CLAIM_TIMEOUT_HOURS) -> None:
    cutoff = time.time() - max_age_hours * 3600
    for f in DRIVE_CLAIMED.glob("*.md"):
        try:
            if f.stat().st_mtime < cutoff:
                os.replace(f, DRIVE_PENDING / f.name)
                print(f"[System] Re-queued stale claim from an interrupted run: {f.name}")
        except OSError:
            pass

def claim_next_query():
    """Moves the oldest pending query into claimed/ and returns (query_text, claimed_path)."""
    for pending in sorted(DRIVE_PENDING.glob("*.md"), key=lambda p: p.name):
        claimed = DRIVE_CLAIMED / pending.name
        try:
            _retry(lambda: os.rename(pending, claimed))   # rename fails if someone else claimed it first
        except OSError:
            continue
        try: os.utime(claimed, None)                      # rename keeps the old mtime; stamp the claim time
        except OSError: pass
        query = claimed.read_text(encoding="utf-8").strip()
        if not query:
            os.replace(claimed, DRIVE_DONE / claimed.name)
            continue
        return query, claimed
    return None, None

def finish_query(claimed_path: Path) -> None:
    _retry(lambda: os.replace(claimed_path, DRIVE_DONE / claimed_path.name))

def read_ledger_names(*paths) -> set:
    names = set()
    for p in paths:
        try: names.update(l.strip() for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip())
        except OSError: pass
    return names

# ==============================================================================
# 5b. PUSH TO DRIVE  (additive, so never mirrors a folder over another)
# ==============================================================================
def _append_to_drive(local_path: Path, drive_path: Path, header: str = "") -> None:
    if not local_path.exists(): return
    delta = local_path.read_text(encoding="utf-8")
    if delta.strip():
        prefix = header if not drive_path.exists() else ""
        if drive_path.exists() and drive_path.stat().st_size > 0:
            with open(drive_path, "rb") as f:
                f.seek(-1, os.SEEK_END)
                if f.read(1) != b"\n": prefix = "\n"
        with open(drive_path, "a", encoding="utf-8") as f:
            f.write(prefix + (delta if delta.endswith("\n") else delta + "\n"))
    local_path.unlink()

def flush_to_drive() -> None:
    """Pushes everything this machine produced. Safe to call twice; also recovers an interrupted run."""
    # 1. Downloaded documents: copy as name.part, then rename, so the cloud never ingests a partial file.
    for f in sorted(LOCAL_UNPROCESSED.iterdir()):
        if not f.is_file(): continue
        dest, part = DRIVE_UNPROCESSED / f.name, DRIVE_UNPROCESSED / (f.name + ".part")
        if not dest.exists():
            _retry(lambda: shutil.copy2(f, part))
            _retry(lambda: os.replace(part, dest))
        f.unlink()
    # 2. Forager-only, append-only files: append only what this run added.
    _retry(lambda: _append_to_drive(LOCAL_PAPERS_LEDGER, DRIVE_PAPERS_LEDGER))
    _retry(lambda: _append_to_drive(LOCAL_HARVESTED, DRIVE_HARVESTED,
                                    header="# Harvested Research Concepts & Keywords\n\n"))
    # 3. Session logs.
    for f in LOCAL_QUERIES.glob("forage_*.md"):
        _retry(lambda: shutil.copy2(f, DRIVE_QUERIES / f.name))
        f.unlink()

def cleanup_local() -> None:
    for directory in [LOCAL_WORKSPACE, LOCAL_PERSONALITY]:
        if directory.exists():
            shutil.rmtree(directory)
            print(f"[✓] Deleted local footprint: {directory}")
    print("[System] Shutdown complete. Laptop storage reclaimed.")

# ==============================================================================
# 6. API CLASSES (ACADEMIC & OPEN WEB)
# ==============================================================================
class AcademicForager:
    def __init__(self, output_dir: Path, ledger_path: Path, session_log: list, source_name: str):
        self.output_dir = output_dir
        self.ledger_path = ledger_path
        self.session_log = session_log
        self.source_name = source_name
        self.anchor = ""
        self.session = requests.Session()
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        ]
        self.session.headers.update({'User-Agent': random.choice(user_agents), 'Accept': 'application/pdf, text/html, */*'})
        if not self.ledger_path.exists(): self.ledger_path.touch()

    def is_already_processed(self, safe_filename: str) -> bool:
        if safe_filename in read_ledger_names(DRIVE_PAPERS_LEDGER, self.ledger_path): return True   # exact match, not substring
        if (self.output_dir / safe_filename).exists() or (DRIVE_UNPROCESSED / safe_filename).exists(): return True
        return False

    def download_document(self, url: str, raw_title: str, fallback_abstract: str = "", year=None) -> bool:
        if not url and not fallback_abstract: return False
        safe_title = generate_safe_title(raw_title)
        safe_pdf, safe_md = f"{safe_title}.pdf", f"{safe_title}.md"

        if not year_in_range(year):     # not written to the ledger, so a wider YEAR window can pick it up later
            y0, y1 = year_bounds()
            self.session_log.append(f"- **[SKIPPED]** *{safe_title}* (year {parse_year(year)} outside {y0}-{y1} | Source: {self.source_name})")
            return False

        if self.is_already_processed(safe_pdf) or self.is_already_processed(safe_md):
            self.session_log.append(f"- **[SKIPPED]** *{safe_title}* (Source: {self.source_name})")
            return False

        success = False
        if url:
            try:
                print(f"    [+] Fetching: {safe_title[:50]}...")
                time.sleep(random.uniform(1.5, 3.5)) 
                response = self.session.get(url, stream=True, timeout=20, allow_redirects=True)
                
                if response.status_code == 200:
                    content_type = response.headers.get('Content-Type', '').lower()
                    if 'application/pdf' in content_type:
                        with open(self.output_dir / safe_pdf, 'wb') as f: f.write(response.content)
                        success = True
                    elif 'text/html' in content_type or 'text/plain' in content_type:
                        dl_html = trafilatura.fetch_url(url)
                        if dl_html:
                            clean_text = trafilatura.extract(dl_html, include_comments=False)
                            if clean_text and len(clean_text) > 300:
                                (self.output_dir / safe_md).write_text(f"# {raw_title}\n\n{clean_text}", encoding="utf-8")
                                success = True
            except Exception as e:
                print(f"    [!] Fetch error for {safe_title[:30]}: {e}")

        if not success and fallback_abstract and len(fallback_abstract) > 100:
            print(f"    [*] Reverting to API Abstract for: {safe_title[:40]}...")
            (self.output_dir / safe_md).write_text(f"# {raw_title}\n\n## Abstract\n{fallback_abstract}", encoding="utf-8")
            success = True

        if success:
            with open(self.ledger_path, 'a', encoding='utf-8') as f: f.write(safe_pdf + '\n' + safe_md + '\n')
            self.session_log.append(f"- **[SAVED]** *{safe_title}* (Source: {self.source_name})")
        else:
            self.session_log.append(f"- **[FAILED]** *{safe_title}* (Source: {self.source_name} | No readable text)")
        return success

    def perform_search_with_retries(self, url: str, custom_headers: dict = None):
        active_headers = custom_headers if custom_headers else self.session.headers
        for attempt in range(3):
            try:
                time.sleep(random.uniform(2.0, 4.0))
                res = self.session.get(url, headers=active_headers, timeout=25)
                if res.status_code in [403, 429, 502, 503, 504]:
                    wait_time = random.uniform(10.0, 20.0) * (attempt + 1)
                    print(f"    [*] {self.source_name} rate limit. Jittering for {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    continue
                if not res.ok: return None
                return res
            except Exception:
                time.sleep(5)
        return None

    def build_query(self, raw_query: str, max_words: int = 4, operator: str = " ", shuffle: bool = True, group_mode: bool = False) -> str:
        words = list(dict.fromkeys(raw_query.replace('"', '').replace("'", "").lower().split()))
        words = [w for w in words if w != self.anchor]     # the anchor is added once, at the front
        if shuffle: random.shuffle(words)
        if self.anchor: words = [self.anchor] + words
        words = words[:max_words]
        if not words: return ""
        if group_mode and len(words) > 1:
            if self.anchor: return f"{words[0]} + ({' | '.join(words[1:])})"          # anchor required, plus any one other word
            return f"({' | '.join(words[0::2])}) + ({' | '.join(words[1::2])})"
        return operator.join(words)

# --- OPEN INTERNET EXPLORER ---
class DuckDuckGoAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        # We don't truncate the query for the open web, we pass the raw bag of words
        # The open web has no reliable publication year, so the YEAR window cannot be applied here.
        print(f"[System] Querying DuckDuckGo (Open Web) for: {query}")
        try:
            results = DDGS().text(query, max_results=limit)
            if not results: return
            for res in results:
                title = res.get('title', 'Unknown_Web_Title')
                url = res.get('href')
                abstract = res.get('body', '')
                self.download_document(url, title, fallback_abstract=abstract)
                time.sleep(random.uniform(2.0, 4.0))
        except Exception as e:
            print(f"    [!] DuckDuckGo search failed: {e}")

# --- ACADEMIC APIs ---
class ArxivAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        arxiv_query = self.build_query(query, max_words=2, operator=" AND ", shuffle=True)
        print(f"[System] Querying arXiv for: {arxiv_query}")
        if not arxiv_query: return
        y0, y1 = year_bounds()
        terms = " AND ".join(f"all:{w}" for w in arxiv_query.split(" AND "))
        search_query = f"{terms} AND submittedDate:[{y0}01010000 TO {y1}12312359]"   # arXiv needs an explicit YYYYMMDDHHMM range
        url = f"http://export.arxiv.org/api/query?search_query={urllib.parse.quote(search_query)}&start=0&max_results={limit}&sortBy=submittedDate&sortOrder=descending"
        res = self.perform_search_with_retries(url)
        if not res: return
        root = ET.fromstring(res.content)
        for entry in root.findall('{http://www.w3.org/2005/Atom}entry'):
            title = entry.find('{http://www.w3.org/2005/Atom}title').text.replace('\n', '')
            abstract = entry.find('{http://www.w3.org/2005/Atom}summary').text.replace('\n', ' ')
            pdf_url = next((link.attrib.get('href') + ".pdf" for link in entry.findall('{http://www.w3.org/2005/Atom}link') if link.attrib.get('title') == 'pdf'), None)
            published = entry.find('{http://www.w3.org/2005/Atom}published')
            self.download_document(pdf_url, title, fallback_abstract=abstract, year=published.text if published is not None else None)

class CoreAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        if not CORE_API_KEY:
            print("[*] Skipping CORE: no CORE_API_KEY in API_KEYS.md"); return
        core_query = self.build_query(query, max_words=3, operator=" ", shuffle=True)
        print(f"[System] Querying CORE for: {core_query}")
        if not core_query: return
        y0, y1 = year_bounds()
        encoded_query = urllib.parse.quote(f'({core_query}) AND yearPublished>="{y0}" AND yearPublished<="{y1}"')
        safe_limit = min(limit, 100)
        base_url = f"https://api.core.ac.uk/v3/search/works?q={encoded_query}&limit={safe_limit}"
        core_headers = self.session.headers.copy()
        core_headers["Authorization"] = f"Bearer {CORE_API_KEY}"
        offset = 0
        successful_yield = 0
        while successful_yield < limit:
            current_url = f"{base_url}&offset={offset}"
            res = self.perform_search_with_retries(current_url, custom_headers=core_headers)
            if not res: break 
            data = res.json()
            works = data.get('results', [])
            if not works: break 
            for work in works:
                if successful_yield >= limit: break
                title = work.get('title', 'Unknown_Title')
                safe_title = generate_safe_title(title)
                if self.is_already_processed(f"{safe_title}.pdf") or self.is_already_processed(f"{safe_title}.md"): continue 
                download_url = work.get('downloadUrl')
                abstract = work.get('abstract', '')
                if self.download_document(download_url, title, fallback_abstract=abstract, year=work.get('yearPublished')): successful_yield += 1 
            if successful_yield >= limit: break
            offset += len(works)

class ZenodoAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        zenodo_query = self.build_query(query, max_words=3, operator=" ", shuffle=True)
        print(f"[System] Querying Zenodo for: {zenodo_query}")
        
        y0, y1 = year_bounds()
        encoded_query = urllib.parse.quote(f"({zenodo_query}) AND publication_date:[{y0}-01-01 TO {y1}-12-31]")
        # Integrate the Zenodo API key directly into the request URL
        token = f"&access_token={ZENODO_API_KEY}" if ZENODO_API_KEY else ""
        url = f"https://zenodo.org/api/records?q={encoded_query}&size={limit}&sort=mostrecent{token}"
        
        # Force JSON response to bypass HTML captive portals
        zenodo_headers = self.session.headers.copy()
        zenodo_headers['Accept'] = 'application/json'
        
        res = self.perform_search_with_retries(url, custom_headers=zenodo_headers)
        if not res: return
        
        # Safely parse the JSON to prevent fatal crashes
        try:
            hits = res.json().get('hits', {}).get('hits', [])
        except Exception:
            print("    [!] Zenodo API returned an invalid (non-JSON) response. Skipping.")
            return
            
        for work in hits:
            title = work.get('metadata', {}).get('title', 'Unknown_Title')
            abstract = work.get('metadata', {}).get('description', '')
            files = work.get('files', [])
            target_url = files[0].get('links', {}).get('self') if files else work.get('links', {}).get('html')
            
            # Pass positionally to match the AcademicForager signature
            self.download_document(target_url, title, fallback_abstract=abstract, year=work.get('metadata', {}).get('publication_date'))

class OpenAlexAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        oa_query = self.build_query(query, max_words=2, operator=" AND ", shuffle=True)
        print(f"[System] Querying OpenAlex for: {oa_query}")
        y0, y1 = year_bounds()
        key = f"&api_key={OPENALEX_API_KEY}" if OPENALEX_API_KEY else ""
        url = f"https://api.openalex.org/works?search={urllib.parse.quote(oa_query)}&filter=from_publication_date:{y0}-01-01,to_publication_date:{y1}-12-31&per-page={limit}{key}"
        res = self.perform_search_with_retries(url)
        if not res: return
        for work in res.json().get('results', []):
            title = work.get('title', 'Unknown_Title')
            oa_url = work.get('open_access', {}).get('oa_url')
            landing_url = work.get('primary_location', {}).get('landing_page_url')
            abstract = reconstruct_abstract(work.get('abstract_inverted_index'))
            harvest_metadata_concepts(work, "OpenAlex")
            self.download_document(oa_url if oa_url else landing_url, title, fallback_abstract=abstract, year=work.get('publication_year'))

class SemanticScholarAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        s2_query = self.build_query(query, max_words=4, shuffle=True, group_mode=True)   # S2 bulk syntax: + = AND, | = OR
        print(f"[System] Querying Semantic Scholar (Bulk) for: {s2_query}")
        encoded_query = urllib.parse.quote(s2_query)
        fields = "title,openAccessPdf,url,abstract,fieldsOfStudy,year"
        y0, y1 = year_bounds()
        base_url = f"https://api.semanticscholar.org/graph/v1/paper/search/bulk?query={encoded_query}&fields={fields}&year={y0}-{y1}"
        s2_headers = self.session.headers.copy()
        if S2_API_KEY: s2_headers["x-api-key"] = S2_API_KEY
        current_url = base_url
        successful_yield = 0
        while successful_yield < limit:
            res = self.perform_search_with_retries(current_url, custom_headers=s2_headers)
            if not res: break
            data = res.json()
            papers = data.get('data', [])
            if not papers: break
            for paper in papers:
                if successful_yield >= limit: break
                title = paper.get('title', 'Unknown_Title')
                safe_title = generate_safe_title(title)
                if self.is_already_processed(f"{safe_title}.pdf") or self.is_already_processed(f"{safe_title}.md"): continue
                oa_url = paper.get('openAccessPdf', {}).get('url') if paper.get('openAccessPdf') else None
                target_url = oa_url if oa_url else paper.get('url')
                abstract = paper.get('abstract', '')
                harvest_metadata_concepts(paper, "Semantic Scholar")
                if self.download_document(target_url, title, fallback_abstract=abstract, year=paper.get('year')): successful_yield += 1
            if successful_yield >= limit: break
            token = data.get('token')
            if not token: break
            current_url = f"{base_url}&token={token}"

class OsfAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        osf_query = self.build_query(query, max_words=1, operator=" ", shuffle=True)
        print(f"    [System] Querying OSF Preprints for: {osf_query}")
        encoded_query = urllib.parse.quote(osf_query)
        safe_limit = min(limit, 100)
        url = f"https://api.osf.io/v2/preprints/?filter[title]={encoded_query}&page[size]={safe_limit}"
        
        # 1. Force the API to return JSON data, not an HTML webpage
        osf_headers = self.session.headers.copy()
        osf_headers['Accept'] = 'application/json'
        
        res = self.perform_search_with_retries(url, custom_headers=osf_headers)
        if not res: return
        
        # 2. Safely attempt to parse the response, catching HTML/Cloudflare pages
        try:
            data = res.json().get('data', [])
        except Exception:
            print("    [!] OSF API returned an invalid (non-JSON) response. Skipping.")
            return
            
        if not data: return
        for work in data:
            attrs = work.get('attributes', {})
            title = attrs.get('title', 'Unknown_Title')
            target_url = work.get('links', {}).get('html')
            if not target_url: continue
            abstract = attrs.get('description', '')
            
            # OSF preprints have no date filter in the API, so the window is enforced here
            self.download_document(target_url, title, fallback_abstract=abstract, year=attrs.get('date_published') or attrs.get('date_created'))

class DoajAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        doaj_query = self.build_query(query, max_words=2, operator=" AND ", shuffle=True)
        print(f"[System] Querying DOAJ for: {doaj_query}")
        y0, y1 = year_bounds()
        encoded_query = urllib.parse.quote(f"({doaj_query}) AND bibjson.year:[{y0} TO {y1}]", safe="")   # DOAJ rejects open-ended (*) ranges
        url = f"https://doaj.org/api/v4/search/articles/{encoded_query}?pageSize={limit}"
        res = self.perform_search_with_retries(url)
        if not res: return
        try: data = res.json()
        except Exception: return
        for work in data.get('results', []):
            bibjson = work.get('bibjson', {})
            links = bibjson.get('link', [])
            target_url = links[0].get('url') if links else None
            self.download_document(target_url, bibjson.get('title', 'Unknown_Title'), fallback_abstract=bibjson.get('abstract', ''), year=bibjson.get('year'))

class HalAPI(AcademicForager):
    def search(self, query: str, limit: int = 3):
        # Solr prefers explicit AND operators for precision
        hal_query = self.build_query(query, max_words=2, operator=" AND ", shuffle=True)
        print(f"[System] Querying HAL Archives for: {hal_query}")
        
        encoded_query = urllib.parse.quote(hal_query)
        # fq=submitType_s:file ensures HAL only returns records that actually have a document attached
        y0, y1 = year_bounds()
        year_fq = urllib.parse.quote(f"producedDateY_i:[{y0} TO {y1}]")
        url = f"https://api.archives-ouvertes.fr/search/?q={encoded_query}&rows={limit}&fl=title_s,abstract_s,fileMain_s,halId_s,producedDateY_i&fq=submitType_s:file&fq={year_fq}&wt=json"
        
        # Force the API to return JSON data, bypassing HTML captive portals
        hal_headers = self.session.headers.copy()
        hal_headers['Accept'] = 'application/json'
        
        res = self.perform_search_with_retries(url, custom_headers=hal_headers)
        if not res: return
        
        # Safely parse the response, catching HTML errors
        try:
            data = res.json().get('response', {}).get('docs', [])
        except Exception:
            print("    [!] HAL API returned an invalid (non-JSON) response. Skipping.")
            return
            
        for work in data:
            title = work.get('title_s', ['Unknown_Title'])[0]
            abstract = work.get('abstract_s', [''])[0] if work.get('abstract_s') else ""
            target_url = work.get('fileMain_s') or f"https://hal.science/{work.get('halId_s')}"
            
            # Pass positionally to match the AcademicForager signature
            self.download_document(target_url, title, fallback_abstract=abstract, year=work.get('producedDateY_i'))
# ==============================================================================
# 7. FORAGING LOOP EXECUTION
# ==============================================================================
def forage_query(active_query: str) -> int:
    """Runs one query through every source, writes its session log, and returns how many documents were saved."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    ledger_path = LOCAL_PAPERS_LEDGER   # this run's additions only; Drive holds the history
    forage_log_path = LOCAL_QUERIES / f"forage_{timestamp}.md"
    nudge_words = load_nudge_words(DRIVE_NUDGE)
    anchor = random.choice(nudge_words) if nudge_words else ""
    print(f"[System] Anchor word: {anchor or 'none (NUDGE.md missing or empty)'}")
    session_log = [
        f"# Foraging Session: {timestamp}\n",
        f"**Query retreived from cloud:** `{active_query}`\n",
        f"**Anchor word:** `{anchor or 'none'}`\n",
        "## Retrieval Results\n"
    ]

    foragers = [
        DuckDuckGoAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "DuckDuckGo"),
        OpenAlexAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "OpenAlex"),
        SemanticScholarAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "Semantic Scholar"),
        ArxivAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "arXiv"),
        CoreAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "CORE"),
        ZenodoAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "Zenodo"),
        OsfAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "OSF Preprints"),
        HalAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "HAL Archives"),
        DoajAPI(LOCAL_UNPROCESSED, ledger_path, session_log, "DOAJ")
    ]

    for forager in foragers:
        forager.anchor = anchor

    for forager in foragers:
        forager.search(active_query, limit=LIMIT)

    forage_log_path.write_text("\n".join(session_log), encoding="utf-8")
    print(f"[System] Foraging session logged to {forage_log_path.name}")
    return sum("[SAVED]" in line for line in session_log)

def run_forager():
    print("[System] Connecting to Drive ...")

    for filename in ["INSTRUCTIONS.md", "NUDGE.md", "MEMORY.md"]:
        src_file = DRIVE_PERSONALITY / filename
        if src_file.exists(): shutil.copy2(src_file, LOCAL_PERSONALITY / filename)

    requeue_stale_claims()
    flush_to_drive()   # recovery: push anything left behind by an interrupted earlier run

    y0, y1 = year_bounds()
    print(f"[System] Publication window: {y0}-{y1}")

    completed = failed = saved_total = 0
    while True:   # one query per pass; stops only when the queue holds no more query files
        if MAX_QUERIES_PER_RUN and completed + failed >= MAX_QUERIES_PER_RUN:
            print(f"[System] Reached MAX_QUERIES_PER_RUN ({MAX_QUERIES_PER_RUN}); stopping with queries still pending.")
            break

        print("[System] Checking for new search queries ...")
        active_query, claimed_path = claim_next_query()
        if not active_query:
            print("[System] Queue is empty.")
            break

        print(f"\n[System] === Query {completed + failed + 1} more pending: {active_query} ===")
        try:
            saved_total += forage_query(active_query)
            flush_to_drive()                 # results first ...
            finish_query(claimed_path)       # ... then mark the query done (a crash before this re-queues it)
            completed += 1
            print("[System] Sync to G drive complete.")
        except Exception as e:
            failed += 1
            print(f"[!] Query failed: {e}")
            print(f"    It stays in claimed/ and returns to the queue after {CLAIM_TIMEOUT_HOURS}h. Moving on to the next query.")
            try: flush_to_drive()            # keep whatever was downloaded before the failure
            except Exception as flush_error: print(f"[!] Could not push partial results yet: {flush_error}")

    print(f"\n[System] Finished: {completed} queries completed, {failed} failed, {saved_total} documents saved.")

    # ==============================================================================
    # 8. SHUTDOWN PROTOCOL: LOCAL CLEANUP
    # ==============================================================================
    cleanup_local()

if __name__ == "__main__":
    run_forager()

[System] Connecting to Drive ...
[System] Publication window: 2021-2027
[System] Checking for new search queries ...
[System] Queue is empty.

[System] Finished: 0 queries completed, 0 failed, 0 documents saved.
[✓] Deleted local footprint: C:\Users\aapre\Desktop\research-assistant\phdresearch-workspace
[✓] Deleted local footprint: C:\Users\aapre\Desktop\research-assistant\phdresearch-personality
[System] Shutdown complete. Laptop storage reclaimed.
